# Training models in one collection date to infer on others

## Loading the Database

In [1]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import numpy as np

In [2]:
IMG_SIZE = 512
BATCH_SIZE = 8
NUM_CLASSES = 1
CROP = "corn"  # "wheat", "sorghum" or "corn"
COLLECTION_DATE = "2"

dataset_path = f"../data/{CROP}/"

class SegmentationDataset(Dataset):
    def __init__(self, root_dir, train=True, collection_dates=None):
        self.root_dir = root_dir
        self.train = train
        self.img_dir = os.path.join(root_dir, "train/images" if train else "test/images")
        self.mask_dir = os.path.join(root_dir, "train/masks" if train else "test/masks")

        # List all image files
        image_files = sorted([
            f for f in os.listdir(self.img_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
        mask_files = sorted([
            f for f in os.listdir(self.mask_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
        
        # Filter by collection date if provided
        if collection_dates is not None:
            # Accept single date or list of dates
            if isinstance(collection_dates, (str, int)):
                collection_dates = [str(collection_dates)]
            image_files = [f for f in image_files if f.split('_')[0] in collection_dates]
            mask_files = [f for f in mask_files if f.split('_')[0] in collection_dates]
        print(image_files)
        # Ensure matching images/masks
        assert len(image_files) == len(mask_files), \
            f"Image/Mask count mismatch: {len(image_files)} vs {len(mask_files)}"

        self.image_files = image_files
        self.mask_files = mask_files

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.image_files[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_files[idx])

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # grayscale mask

        image = TF.to_tensor(image)
        image = TF.normalize(image, mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        mask = torch.from_numpy(np.array(mask)).float().unsqueeze(0)
        mask = (mask > 0).float()

        return image, mask

train_dataset = SegmentationDataset(dataset_path, train=True, collection_dates=COLLECTION_DATE)
test_dataset = SegmentationDataset(dataset_path, train=False, collection_dates=COLLECTION_DATE)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"✅ Train size: {len(train_dataset)} | Test size: {len(test_dataset)}")

['2_100_1.jpg', '2_100_1_aug1.jpg', '2_100_1_aug2.jpg', '2_100_1_aug3.jpg', '2_100_1_aug4.jpg', '2_100_1_aug5.jpg', '2_100_2.jpg', '2_100_2_aug1.jpg', '2_100_2_aug2.jpg', '2_100_2_aug3.jpg', '2_100_2_aug4.jpg', '2_100_2_aug5.jpg', '2_100_3.jpg', '2_100_3_aug1.jpg', '2_100_3_aug2.jpg', '2_100_3_aug3.jpg', '2_100_3_aug4.jpg', '2_100_3_aug5.jpg', '2_101_1.jpg', '2_101_1_aug1.jpg', '2_101_1_aug2.jpg', '2_101_1_aug3.jpg', '2_101_1_aug4.jpg', '2_101_1_aug5.jpg', '2_101_2.jpg', '2_101_2_aug1.jpg', '2_101_2_aug2.jpg', '2_101_2_aug3.jpg', '2_101_2_aug4.jpg', '2_101_2_aug5.jpg', '2_101_3.jpg', '2_101_3_aug1.jpg', '2_101_3_aug2.jpg', '2_101_3_aug3.jpg', '2_101_3_aug4.jpg', '2_101_3_aug5.jpg', '2_102_1.jpg', '2_102_1_aug1.jpg', '2_102_1_aug2.jpg', '2_102_1_aug3.jpg', '2_102_1_aug4.jpg', '2_102_1_aug5.jpg', '2_102_2.jpg', '2_102_2_aug1.jpg', '2_102_2_aug2.jpg', '2_102_2_aug3.jpg', '2_102_2_aug4.jpg', '2_102_2_aug5.jpg', '2_102_3.jpg', '2_102_3_aug1.jpg', '2_102_3_aug2.jpg', '2_102_3_aug3.jpg', '2_1

In [3]:
for i in range(3):
    img, mask = train_dataset[i]
    print(img.shape, mask.shape)

torch.Size([3, 512, 512]) torch.Size([1, 512, 512])
torch.Size([3, 512, 512]) torch.Size([1, 512, 512])
torch.Size([3, 512, 512]) torch.Size([1, 512, 512])


## Importing Models

In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


### U-NET

In [4]:
import sys
sys.path.append('../')
from utils.models.uNet import UNet

channels = [32, 64, 128, 256, 512]
model = UNet(in_channels=3, out_channels=1, channels=channels, bilinear=True, use_batchnorm=True)
model.to(device)
modelName = "U-NET"

### SegFormer

In [4]:
import sys
sys.path.append('../')
from utils.models.SegFormer import segformer

model = segformer(in_channels = 3, num_classes = 1)
model.to(device)
modelName = "SegFormer"

c:\Users\gnoceras\Documents\SegmentationStudy\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\gnoceras\Documents\SegmentationStudy\.venv\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [5]:
import torch.nn.functional as F
from transformers import SegformerForSemanticSegmentation

class SegFormerWrapper(torch.nn.Module):
    def __init__(self, num_labels=1):
        super().__init__()
        self.model = SegformerForSemanticSegmentation.from_pretrained(
            "nvidia/mit-b0",
            num_labels=num_labels,
            ignore_mismatched_sizes=True,
        )

    def forward(self, x):
        h, w = x.shape[-2:]
        logits = self.model(pixel_values=x).logits  # [B, num_labels, H/4, W/4]
        return F.interpolate(logits, size=(h, w), mode="bilinear", align_corners=False)

model = SegFormerWrapper(num_labels=1).to(device)
modelName = "SegFormer"

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b0 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Training

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import os
import copy
from tqdm import tqdm
import optuna
from torch.utils.data import Subset

In [9]:
# --- Optuna: Bayesian Hyperparameter Optimization ---
# Uses 1/5 of the training data and runs one full epoch per trial.

optuna_size = len(train_dataset) // 5
optuna_indices = list(range(0, len(train_dataset), 5))[:optuna_size]
optuna_subset = Subset(train_dataset, optuna_indices)
optuna_loader = DataLoader(optuna_subset, batch_size=BATCH_SIZE, shuffle=True)

# Snapshot initial weights so every trial starts from the same state
_initial_weights = copy.deepcopy(model.state_dict())

def _optuna_objective(trial):
    model.load_state_dict(copy.deepcopy(_initial_weights))

    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"])

    if optimizer_name == "SGD":
        momentum = trial.suggest_float("momentum", 0.8, 0.99)
        opt = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay, momentum=momentum)
    elif optimizer_name == "AdamW":
        opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        opt = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    fn = nn.BCEWithLogitsLoss()
    model.train()
    epoch_loss = 0.0
    for inputs, labels in optuna_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        opt.zero_grad()
        outputs = model(inputs)
        loss = fn(outputs, labels)
        loss.backward()
        opt.step()
        epoch_loss += loss.item()

    return epoch_loss / len(optuna_loader)

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler())
study.optimize(_optuna_objective, n_trials=20, show_progress_bar=True)

# Restore model to initial weights before real training
model.load_state_dict(copy.deepcopy(_initial_weights))

best_params = study.best_params
print(f"\nBest hyperparameters: {best_params}")
print(f"Best trial loss:       {study.best_value:.4f}")


Best trial: 14. Best value: 0.247566: 100%|██████████| 20/20 [09:00<00:00, 27.03s/it]


Best hyperparameters: {'lr': 0.009593916616100588, 'weight_decay': 7.978375421706759e-05, 'optimizer': 'SGD', 'momentum': 0.9079477840611434}
Best trial loss:       0.2476


In [10]:
# --- Configuration ---
EPOCHS = 200
early_stopping_patience = 5
best_val_loss = float('inf')
patience_counter = 0
scaler = torch.amp.GradScaler('cuda')

# --- Loss ---
loss_fn = nn.BCEWithLogitsLoss()

# --- Optimizer from Optuna best params ---
_opt_name = best_params.get("optimizer", "Adam")
_lr       = best_params.get("lr", 1e-3)
_wd       = best_params.get("weight_decay", 1e-4)

if _opt_name == "Adam":
    optimizer = optim.Adam(model.parameters(), lr=_lr, weight_decay=_wd)
elif _opt_name == "AdamW":
    optimizer = optim.AdamW(model.parameters(), lr=_lr, weight_decay=_wd)
else:
    _momentum = best_params.get("momentum", 0.9)
    optimizer = optim.SGD(model.parameters(), lr=_lr, weight_decay=_wd, momentum=_momentum)

lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)

checkpoint_path = os.path.join("../models/", CROP + "_" + modelName + "_date" + COLLECTION_DATE + "_seg_rev.pt")
print(f"Model checkpoints will be saved to: {checkpoint_path}")
print(f"Optimizer: {_opt_name} | lr={_lr:.2e} | weight_decay={_wd:.2e}")

# --- Tracking ---
train_losses, val_losses = [], []


Model checkpoints will be saved to: ../models/corn_SegFormer_date2_seg_rev.pt
Optimizer: SGD | lr=9.59e-03 | weight_decay=7.98e-05


In [11]:
# --- Training Loop ---
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    with tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", unit="batch") as tepoch:
        for inputs, labels in tepoch:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                outputs = model(inputs)
                if torch.isnan(outputs).any() or torch.isinf(outputs).any():
                    print("NaN or Inf in model outputs!")
                if torch.isnan(labels).any() or torch.isinf(labels).any():
                    print("NaN or Inf in labels!")
                if labels.sum() == 0:
                    print("Skipping batch with all empty masks")
                    continue
                loss = loss_fn(outputs, labels)
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            tepoch.set_postfix(loss=running_loss / (len(tepoch)))
    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # --- Validation Phase ---
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        with tqdm(test_loader, desc=f"Epoch {epoch+1}/{EPOCHS} - Validation", unit="batch") as vepoch:
            for inputs, labels in vepoch:
                inputs, labels = inputs.to(device), labels.to(device)
                with torch.amp.autocast('cuda'):
                    outputs = model(inputs)
                    if torch.isnan(outputs).any() or torch.isinf(outputs).any():
                        print("NaN or Inf in model outputs!")
                    if torch.isnan(labels).any() or torch.isinf(labels).any():
                        print("NaN or Inf in labels!")
                    if labels.sum() == 0:
                        print("Skipping batch with all empty masks")
                        continue
                    loss = loss_fn(outputs, labels)
                    preds = (torch.sigmoid(outputs) > 0.5).float()
                val_correct += (preds == labels).sum().item()
                val_total += labels.numel()
                val_acc = val_correct / val_total if val_total > 0 else 0
                val_loss += loss.item()
                vepoch.set_postfix(loss=val_loss / (len(vepoch)))

    avg_val_loss = val_loss / len(test_loader)
    val_losses.append(avg_val_loss)

    print(f"Epoch {epoch+1} - Val Loss: {avg_val_loss:.4f} - Val Acc: {val_acc:.4f}%")

    # --- Scheduler step (use validation loss) ---
    lr_scheduler.step(avg_val_loss)

    # --- Checkpointing ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), checkpoint_path)
        print(f"✅ Model improved (Val Loss={avg_val_loss:.3f}), saved to {checkpoint_path}")
    else:
        patience_counter += 1
        print(f"🔴 No improvement, patience counter: {patience_counter}")

    if patience_counter >= early_stopping_patience:
        print("🛑 Early stopping triggered!")
        break

Epoch 1/200 - Validation: 100%|██████████| 120/120 [00:16<00:00,  7.06batch/s, loss=0.684]


Epoch 1 - Val Loss: 0.6839 - Val Acc: 0.5668%
✅ Model improved (Val Loss=0.684), saved to ../models/corn_SegFormer_date2_seg_rev.pt


Epoch 2/200 - Validation: 100%|██████████| 120/120 [00:15<00:00,  7.94batch/s, loss=0.681]


Epoch 2 - Val Loss: 0.6809 - Val Acc: 0.5824%
✅ Model improved (Val Loss=0.681), saved to ../models/corn_SegFormer_date2_seg_rev.pt


Epoch 3/200 - Validation: 100%|██████████| 120/120 [00:14<00:00,  8.14batch/s, loss=0.679]


...

Epoch 32 - Val Loss: 0.6657 - Val Acc: 0.6988%
🔴 No improvement, patience counter: 3


Epoch 33/200 - Validation: 100%|██████████| 120/120 [00:15<00:00,  7.67batch/s, loss=0.664]


Epoch 33 - Val Loss: 0.6638 - Val Acc: 0.7070%
🔴 No improvement, patience counter: 4


Epoch 34/200 - Validation: 100%|██████████| 120/120 [00:15<00:00,  7.84batch/s, loss=0.661]

Epoch 34 - Val Loss: 0.6608 - Val Acc: 0.7337%
🔴 No improvement, patience counter: 5
🛑 Early stopping triggered!
